# Notebook 01 — Target Preparation & Structural Analysis

**Phase 1** of the Protein Binder Evaluation & RL-Guided Design Pipeline.

This notebook:
1. Downloads PDB structures for benchmark targets (IL-7Rα, EGFR)
2. Parses structures with BioPython — chains, residues, secondary structure (DSSP)
3. Identifies hotspot residues at the binding interface
4. Visualises targets in 3D using py3Dmol
5. Computes sequence properties (MW, charge, hydrophobicity)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.target_prep import TargetProtein, BENCHMARK_TARGETS, load_benchmark_targets

print('Benchmark targets available:', list(BENCHMARK_TARGETS.keys()))

## 1.1 Download & Parse Target Structures

In [ ]:
# Load two targets to start: IL-7Rα and EGFR
targets = {}

for name in ['EGFR', 'IL7RA']:
    meta = BENCHMARK_TARGETS[name]
    tp = TargetProtein(
        pdb_id=meta['pdb_id'],
        chain_id=meta['chain_id'],
    )
    tp.download_pdb()
    tp.parse_structure()
    tp.identify_hotspots()
    targets[name] = tp
    print(tp)

## 1.2 Residue-Level Data

In [ ]:
egfr = targets['EGFR']
df = egfr.to_dataframe()
print(f'EGFR: {len(df)} residues, {df["is_hotspot"].sum()} hotspot residues')
df.head(10)

In [ ]:
# Secondary structure composition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, target) in zip(axes, targets.items()):
    df_t = target.to_dataframe()
    
    # DSSP secondary structure counts
    ss_map = {'H': 'Helix', 'E': 'Strand', 'C': 'Coil', '-': 'Unassigned'}
    ss_counts = df_t['dssp_ss'].map(ss_map).fillna('Unassigned').value_counts()
    
    colors = ['#2E86AB', '#E63946', '#F4A261', '#A8DADC']
    ax.pie(ss_counts.values, labels=ss_counts.index, colors=colors,
           autopct='%1.1f%%', startangle=140)
    ax.set_title(f'{name} ({target.pdb_id})\nSecondary Structure', fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/secondary_structure_targets.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure.')

## 1.3 Hotspot Residue Visualisation

In [ ]:
# Hotspot residue distribution along the sequence
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ax, (name, target) in zip(axes, targets.items()):
    df_t = target.to_dataframe()
    hotspot_pos = df_t[df_t['is_hotspot']]['res_seq'].values
    all_pos = df_t['res_seq'].values
    sasa_vals = df_t['sasa'].values
    
    # SASA profile
    ax.fill_between(all_pos, sasa_vals, alpha=0.3, color='#457B9D', label='SASA')
    ax.vlines(hotspot_pos, 0, sasa_vals.max(), 
              colors='#E63946', alpha=0.6, linewidth=1.5, label='Hotspot residues')
    
    ax.set_xlabel('Residue number', fontsize=11)
    ax.set_ylabel('Relative SASA', fontsize=11)
    ax.set_title(f'{name} ({target.pdb_id}): {len(hotspot_pos)} hotspot residues highlighted',
                fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/hotspot_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 3D Visualisation with py3Dmol

In [ ]:
try:
    import py3Dmol
    import pathlib
    
    target = targets['EGFR']
    hotspot_nums = target.hotspot_residue_numbers()
    
    view = py3Dmol.view(width=800, height=500)
    pdb_text = pathlib.Path(str(target.pdb_path)).read_text()
    view.addModel(pdb_text, 'pdb')
    
    # Cartoon representation for whole chain
    view.setStyle({'chain': target.chain_id}, {'cartoon': {'color': 'spectrum'}})
    
    # Highlight hotspot residues as spheres
    if hotspot_nums:
        hotspot_str = '+'.join(str(r) for r in hotspot_nums[:30])  # show first 30
        view.addStyle(
            {'chain': target.chain_id, 'resi': hotspot_str},
            {'sphere': {'color': 'red', 'radius': 0.7}}
        )
    
    view.zoomTo()
    view.show()
    print(f'Displaying EGFR ({target.pdb_id}) with {len(hotspot_nums)} hotspot residues (red spheres)')
    
except ImportError:
    print('py3Dmol not installed. pip install py3Dmol')
    print(f'EGFR hotspot residues: {targets["EGFR"].hotspot_residue_numbers()[:10]}...')

## 1.5 Sequence Properties

In [ ]:
for name, target in targets.items():
    try:
        props = target.compute_sequence_properties()
        print(f'\n--- {name} ({target.pdb_id}) ---')
        for k, v in props.items():
            print(f'  {k:30s}: {v:.3f}' if isinstance(v, float) else f'  {k:30s}: {v}')
    except Exception as e:
        print(f'{name}: Could not compute properties — {e}')

In [ ]:
# Hydrophobicity profile along sequence
kd_scale = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2, 'X': 0.0,
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ax, (name, target) in zip(axes, targets.items()):
    seq = target.sequence
    hydro = [kd_scale.get(aa, 0.0) for aa in seq]
    
    # Sliding window average (window=9)
    w = 9
    smoothed = np.convolve(hydro, np.ones(w)/w, mode='same')
    positions = np.arange(len(seq))
    
    ax.fill_between(positions, smoothed, 0,
                    where=smoothed > 0, color='#E63946', alpha=0.6, label='Hydrophobic')
    ax.fill_between(positions, smoothed, 0,
                    where=smoothed <= 0, color='#2E86AB', alpha=0.6, label='Hydrophilic')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    
    ax.set_xlabel('Residue position', fontsize=11)
    ax.set_ylabel('KD Hydrophobicity', fontsize=11)
    ax.set_title(f'{name}: Hydrophobicity Profile (window={w})', fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/hydrophobicity_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.6 Summary

The `TargetProtein` class cleanly encapsulates:
- PDB download and parsing
- Secondary structure assignment via DSSP
- Interface hotspot identification (distance-based or surface SASA proxy)
- Sequence biophysical properties

Both `EGFR (3NJP)` and `IL-7Rα (3DI2)` are now ready as inputs to Phase 2 binder generation.